# Punctual event detection

Complementa `domain_shift_trend_analysis.ipynb` che rileva trend graduali e regime shift.
Qui si cercano **eventi puntuali di produzione**: mesi isolati con anomalia di PR che poi rientra.

**Metodi:**
1. Plant-level point anomaly (modified Z-score su PR_PVGIS)
2. Month-over-month shock (variazione % vs mese precedente con recovery)
3. Fleet-relative outlier (deviazione dal fleet median nello stesso mese)
4. Fleet co-occurrence (quanti impianti anomali per mese → evento condiviso?)
5. Event catalog con classificazione `action_for_training`

**Classificazione per continual learning.** Ogni evento riceve un'azione:
- `include` — evento ambientale plausibile (es. heatwave, soiling lieve), il modello deve imparare.
- `downweight` — ambiguo, includere nel training ma con peso ridotto.
- `exclude` — quasi certamente equipment failure o data issue, escludere dal training.

La logica usa: severita' del drop (%), fleet co-occurrence (plant-specific vs fleet-wide), e letteratura NREL che indica >25-30% drop come improbabile per cause ambientali.

**Interpretazione.** Un evento puntuale non e' automaticamente un guasto. Possibili cause:
soiling temporaneo, curtailment, inverter trip, ombreggiamento stagionale,
manutenzione, meteo estremo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
OUT_TREND = ROOT / 'outputs' / 'domain_shift_trend'
OUT = ROOT / 'outputs' / 'punctual_events'
OUT.mkdir(parents=True, exist_ok=True)

# --- detection thresholds ---
ZSCORE_THRESHOLD = 2.0          # modified Z-score for point anomaly
MOM_DROP_PCT = -15.0            # month-over-month drop % to flag
MOM_SPIKE_PCT = 30.0            # month-over-month spike % to flag
FLEET_ZSCORE_THRESHOLD = 2.0    # deviation from fleet median
COOCCURRENCE_PCT = 0.05         # fraction of fleet for co-occurrence (5%)
COOCCURRENCE_MIN_FLOOR = 5      # absolute minimum
PR_PLAUSIBLE_MAX = 2.0          # exclude months with PR above this
MIN_VALID_HOURS = 30            # exclude months with fewer valid hours

# --- severity thresholds for training action (based on NREL literature) ---
MODERATE_DROP_PCT = 15.0        # drop % below which = environmental plausible
SEVERE_DROP_PCT = 25.0          # drop % above which = equipment/data issue

## Caricamento dati

Usa gli output gia' generati da `analyze_domain_shift_trend.py`.

In [ ]:
plant_monthly_raw = pd.read_csv(
    OUT_TREND / 'plant_monthly_performance.csv', parse_dates=['date']
)

print(f'plant_monthly shape (raw): {plant_monthly_raw.shape}')
print(f'plants: {plant_monthly_raw["plant"].nunique()}')
print(f'date range: {plant_monthly_raw["date"].min()} — {plant_monthly_raw["date"].max()}')

## Filtri pre-analisi

Tre filtri per ridurre noise da artefatti:
1. **Mesi bordo**: rimuove primo e ultimo mese di ogni impianto (dati parziali → PR distorto).
2. **Valid hours**: rimuove mesi con poche ore valide (stessa causa).
3. **PR plausibili**: rimuove mesi con PR > 2.0 (fisicamente impossibile, artefatto calcolo).

In [ ]:
n_raw = len(plant_monthly_raw)

# 1. Drop edge months per plant (first and last)
plant_monthly_f = plant_monthly_raw.copy()
edge_mask = pd.Series(False, index=plant_monthly_f.index)
for _, g in plant_monthly_f.groupby('plant'):
    if len(g) <= 2:
        edge_mask.loc[g.index] = True
        continue
    sorted_idx = g.sort_values('date').index
    edge_mask.loc[sorted_idx[0]] = True
    edge_mask.loc[sorted_idx[-1]] = True
n_edge = edge_mask.sum()
plant_monthly_f = plant_monthly_f[~edge_mask]

# 2. Drop months with too few valid hours
has_valid_hours = 'valid_hours' in plant_monthly_f.columns
if has_valid_hours:
    low_hours = plant_monthly_f['valid_hours'] < MIN_VALID_HOURS
    n_low_hours = low_hours.sum()
    plant_monthly_f = plant_monthly_f[~low_hours]
else:
    n_low_hours = 0
    print('WARNING: valid_hours column not found, skipping hours filter.')

# 3. Drop implausible PR
implausible = plant_monthly_f['pr_pvgis'] > PR_PLAUSIBLE_MAX
n_implausible = implausible.sum()
plant_monthly_f = plant_monthly_f[~implausible]

plant_monthly = plant_monthly_f.reset_index(drop=True)

# Compute proportional co-occurrence threshold
n_fleet_plants = plant_monthly['plant'].nunique()
COOCCURRENCE_MIN_PLANTS = max(COOCCURRENCE_MIN_FLOOR, int(n_fleet_plants * COOCCURRENCE_PCT))

print(f'Rows raw:         {n_raw}')
print(f'Dropped edge:     {n_edge}')
print(f'Dropped low hrs:  {n_low_hours}')
print(f'Dropped PR>{PR_PLAUSIBLE_MAX}: {n_implausible}')
print(f'Rows after filter: {len(plant_monthly)}')
print(f'Plants remaining:  {n_fleet_plants}')
print(f'Co-occurrence threshold: {COOCCURRENCE_MIN_PLANTS} '
      f'(max({COOCCURRENCE_MIN_FLOOR}, {n_fleet_plants}*{COOCCURRENCE_PCT:.0%}))')

## 1. Plant-level point anomaly (modified Z-score)

Per ogni impianto, calcola modified Z-score su `pr_pvgis` mensile usando mediana e MAD.
Flag mesi con |Z| > soglia. MAD e' robusto a outlier, meglio di std per serie corte.

In [ ]:
def modified_zscore(series: pd.Series) -> pd.Series:
    """Modified Z-score using median and MAD."""
    med = series.median()
    mad = np.median(np.abs(series - med))
    if mad < 1e-9:
        return pd.Series(0.0, index=series.index)
    return 0.6745 * (series - med) / mad


records = []
for (plant, plant_id), g in plant_monthly.groupby(['plant', 'plant_id']):
    pr = g.set_index('date')['pr_pvgis'].dropna().sort_index()
    if len(pr) < 4:
        continue
    z = modified_zscore(pr)
    anomalies = z[z.abs() > ZSCORE_THRESHOLD]
    for dt, zval in anomalies.items():
        records.append({
            'plant': plant,
            'plant_id': plant_id,
            'date': dt,
            'pr_pvgis': pr.loc[dt],
            'plant_median_pr': pr.median(),
            'modified_zscore': zval,
            'event_type': 'drop' if zval < 0 else 'spike',
            'method': 'modified_zscore',
        })

anomaly_zscore = pd.DataFrame(records)
print(f'Point anomalies found: {len(anomaly_zscore)}')
if not anomaly_zscore.empty:
    print(anomaly_zscore.groupby('event_type').size())
    print(f'\nPlants with at least 1 anomaly: {anomaly_zscore["plant"].nunique()}')
    display(anomaly_zscore.sort_values('modified_zscore').head(20))

## 2. Month-over-month shock

Flag mesi dove PR cambia >X% rispetto a mese precedente. Distingue:
- **transient drop**: cala e poi recupera mese dopo
- **transient spike**: sale e poi cala mese dopo
- **sustained drop/spike**: non recupera (gia' catturato da trend analysis, ma utile come catalogo)

In [ ]:
mom_records = []
for (plant, plant_id), g in plant_monthly.groupby(['plant', 'plant_id']):
    pr = g.set_index('date')['pr_pvgis'].dropna().sort_index()
    if len(pr) < 3:
        continue
    pct_change = pr.pct_change() * 100
    for i in range(1, len(pr)):
        chg = pct_change.iloc[i]
        if pd.isna(chg):
            continue
        if chg < MOM_DROP_PCT or chg > MOM_SPIKE_PCT:
            dt = pr.index[i]
            recovery = False
            if i + 1 < len(pr):
                next_chg = pct_change.iloc[i + 1] if i + 1 < len(pct_change) else 0
                if chg < MOM_DROP_PCT and not pd.isna(next_chg) and next_chg > abs(chg) * 0.5:
                    recovery = True
                elif chg > MOM_SPIKE_PCT and not pd.isna(next_chg) and next_chg < -chg * 0.5:
                    recovery = True
            event_type = 'transient_drop' if (chg < 0 and recovery) else \
                         'transient_spike' if (chg > 0 and recovery) else \
                         'sustained_drop' if chg < 0 else 'sustained_spike'
            mom_records.append({
                'plant': plant,
                'plant_id': plant_id,
                'date': dt,
                'pr_pvgis': pr.iloc[i],
                'pr_pvgis_prev': pr.iloc[i - 1],
                'mom_change_pct': chg,
                'recovery': recovery,
                'event_type': event_type,
                'method': 'mom_shock',
            })

anomaly_mom = pd.DataFrame(mom_records)
print(f'MoM shocks found: {len(anomaly_mom)}')
if not anomaly_mom.empty:
    print(anomaly_mom.groupby('event_type').size())
    print(f'\nTransient events: {anomaly_mom["recovery"].sum()}')
    display(anomaly_mom.sort_values('mom_change_pct').head(20))

## 3. Fleet-relative outlier

Per ogni mese, calcola mediana e MAD fleet-wide. Flag impianti che deviano >2 modified-Z
dal comportamento collettivo. Cattura impianti "fuori dal coro" in un mese specifico.

In [ ]:
fleet_records = []
for dt, g in plant_monthly.groupby('date'):
    pr_vals = g['pr_pvgis'].dropna()
    if len(pr_vals) < 5:
        continue
    fleet_med = pr_vals.median()
    fleet_mad = np.median(np.abs(pr_vals - fleet_med))
    if fleet_mad < 1e-9:
        continue
    for _, row in g.iterrows():
        if pd.isna(row['pr_pvgis']):
            continue
        z = 0.6745 * (row['pr_pvgis'] - fleet_med) / fleet_mad
        if abs(z) > FLEET_ZSCORE_THRESHOLD:
            fleet_records.append({
                'plant': row['plant'],
                'plant_id': row['plant_id'],
                'date': dt,
                'pr_pvgis': row['pr_pvgis'],
                'fleet_median_pr': fleet_med,
                'fleet_zscore': z,
                'event_type': 'below_fleet' if z < 0 else 'above_fleet',
                'method': 'fleet_relative',
            })

anomaly_fleet = pd.DataFrame(fleet_records)
print(f'Fleet-relative outliers: {len(anomaly_fleet)}')
if not anomaly_fleet.empty:
    print(anomaly_fleet.groupby('event_type').size())
    print(f'\nPlants flagged: {anomaly_fleet["plant"].nunique()}')
    display(anomaly_fleet.sort_values('fleet_zscore').head(20))

## 4. Fleet co-occurrence

Conta quanti impianti hanno anomalia nello stesso mese (da metodo 1 + metodo 2 transient).
Se >= soglia → possibile evento fleet-wide (meteo estremo, curtailment, data pipeline).

Heatmap: mesi (x) vs impianti (y), colore = severita' anomalia.

In [ ]:
# Merge point anomalies + transient MoM shocks
all_plant_events = []
if not anomaly_zscore.empty:
    all_plant_events.append(
        anomaly_zscore[['plant', 'plant_id', 'date', 'event_type', 'method']].copy()
    )
if not anomaly_mom.empty:
    transient = anomaly_mom[anomaly_mom['recovery']]
    if not transient.empty:
        all_plant_events.append(
            transient[['plant', 'plant_id', 'date', 'event_type', 'method']].copy()
        )

if all_plant_events:
    all_events_df = pd.concat(all_plant_events, ignore_index=True)
    cooccurrence = (
        all_events_df.groupby('date')['plant']
        .nunique()
        .reset_index(name='n_plants_flagged')
        .sort_values('n_plants_flagged', ascending=False)
    )
    fleet_events = cooccurrence[
        cooccurrence['n_plants_flagged'] >= COOCCURRENCE_MIN_PLANTS
    ]
    print(f'Co-occurrence threshold: {COOCCURRENCE_MIN_PLANTS} plants '
          f'({COOCCURRENCE_PCT:.0%} of {n_fleet_plants})')
    print(f'Months with >= {COOCCURRENCE_MIN_PLANTS} plants flagged: {len(fleet_events)}')
    display(cooccurrence.head(15))
else:
    all_events_df = pd.DataFrame()
    cooccurrence = pd.DataFrame()
    fleet_events = pd.DataFrame()
    print('No events to analyze for co-occurrence.')

In [ ]:
if not anomaly_zscore.empty:
    pivot = anomaly_zscore.pivot_table(
        index='plant_id', columns='date', values='modified_zscore',
        aggfunc='first'
    )
    if pivot.shape[0] > 2 and pivot.shape[1] > 1:
        fig, ax = plt.subplots(figsize=(14, max(6, pivot.shape[0] * 0.25)))
        sns.heatmap(
            pivot, cmap='RdBu_r', center=0, ax=ax,
            xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Modified Z-score'},
        )
        ax.set_title('Point anomalies: plant x month heatmap')
        ax.set_xlabel('Month')
        ax.set_ylabel('Plant ID')
        plt.tight_layout()
        fig.savefig(OUT / 'anomaly_heatmap_zscore.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('Not enough data for heatmap.')
else:
    print('No Z-score anomalies for heatmap.')

In [ ]:
if not cooccurrence.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    dates = pd.to_datetime(cooccurrence['date'])
    ax.bar(dates, cooccurrence['n_plants_flagged'], width=25, alpha=0.7)
    ax.axhline(COOCCURRENCE_MIN_PLANTS, color='red', linestyle='--',
               label=f'Fleet event threshold ({COOCCURRENCE_MIN_PLANTS} plants)')
    ax.set_xlabel('Month')
    ax.set_ylabel('N plants flagged')
    ax.set_title('Anomaly co-occurrence timeline')
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUT / 'cooccurrence_timeline.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Event catalog con classificazione per continual learning

Aggrega tutti gli eventi in un CSV unico. Ogni evento riceve `action_for_training`:

| Drop severity | Fleet-wide (co-occurrence)? | Plant-specific? |
|---|---|---|
| < 15% | `include` | `include` |
| 15-25% | `include` (meteo plausibile) | `downweight` |
| > 25% | `downweight` (raro ma possibile) | `exclude` |

Per spike: simmetrico ma con soglie piu' alte (spike fisicamente meno probabile di drop).

In [ ]:
# Build set of fleet-event months for classification
fleet_event_months = set()
if not fleet_events.empty:
    fleet_event_months = set(pd.to_datetime(fleet_events['date']))

def classify_training_action(row):
    """Assign action_for_training based on severity + fleet context."""
    is_fleet = row['date'] in fleet_event_months
    pr_val = row.get('metric_value', np.nan)
    ref_val = row.get('reference_value', np.nan)
    if pd.notna(pr_val) and pd.notna(ref_val) and ref_val > 0:
        deviation_pct = abs((pr_val - ref_val) / ref_val) * 100
    else:
        deviation_pct = row['severity']

    if deviation_pct < MODERATE_DROP_PCT:
        return 'include'
    elif deviation_pct < SEVERE_DROP_PCT:
        return 'include' if is_fleet else 'downweight'
    else:
        return 'downweight' if is_fleet else 'exclude'

catalog_rows = []

# Z-score anomalies
if not anomaly_zscore.empty:
    for _, r in anomaly_zscore.iterrows():
        catalog_rows.append({
            'plant': r['plant'],
            'plant_id': r['plant_id'],
            'date': r['date'],
            'method': 'modified_zscore',
            'event_type': r['event_type'],
            'severity': abs(r['modified_zscore']),
            'metric_value': r['pr_pvgis'],
            'reference_value': r['plant_median_pr'],
            'detail': f'Z={r["modified_zscore"]:.2f}',
        })

# MoM shocks
if not anomaly_mom.empty:
    for _, r in anomaly_mom.iterrows():
        catalog_rows.append({
            'plant': r['plant'],
            'plant_id': r['plant_id'],
            'date': r['date'],
            'method': 'mom_shock',
            'event_type': r['event_type'],
            'severity': abs(r['mom_change_pct']),
            'metric_value': r['pr_pvgis'],
            'reference_value': r['pr_pvgis_prev'],
            'detail': f'MoM={r["mom_change_pct"]:.1f}% recovery={r["recovery"]}',
        })

# Fleet-relative
if not anomaly_fleet.empty:
    for _, r in anomaly_fleet.iterrows():
        catalog_rows.append({
            'plant': r['plant'],
            'plant_id': r['plant_id'],
            'date': r['date'],
            'method': 'fleet_relative',
            'event_type': r['event_type'],
            'severity': abs(r['fleet_zscore']),
            'metric_value': r['pr_pvgis'],
            'reference_value': r['fleet_median_pr'],
            'detail': f'fleet_Z={r["fleet_zscore"]:.2f}',
        })

catalog = pd.DataFrame(catalog_rows)
if not catalog.empty:
    catalog['action_for_training'] = catalog.apply(classify_training_action, axis=1)
    catalog = catalog.sort_values(['date', 'plant', 'method']).reset_index(drop=True)
    catalog.to_csv(OUT / 'event_catalog.csv', index=False)
    print(f'Event catalog: {len(catalog)} events, {catalog["plant"].nunique()} plants')
    print(f'Saved to {OUT / "event_catalog.csv"}')
    print()
    print('=== By method x event_type ===')
    print(catalog.groupby(['method', 'event_type']).size().unstack(fill_value=0))
    print()
    print('=== action_for_training breakdown ===')
    print(catalog['action_for_training'].value_counts())
    print()
    print('=== action_for_training by method ===')
    print(catalog.groupby(['method', 'action_for_training']).size().unstack(fill_value=0))
else:
    print('No events detected.')

## Summary visualizzazioni

In [ ]:
if not catalog.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Events per method
    catalog.groupby('method').size().plot.barh(ax=axes[0], color='steelblue')
    axes[0].set_title('Events per detection method')
    axes[0].set_xlabel('N events')

    # Events per month (all methods)
    monthly_counts = catalog.groupby(catalog['date'].dt.to_period('M')).size()
    monthly_counts.index = monthly_counts.index.to_timestamp()
    axes[1].bar(monthly_counts.index, monthly_counts.values, width=25, alpha=0.7, color='coral')
    axes[1].set_title('Total events per month (all methods)')
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('N events')

    plt.tight_layout()
    fig.savefig(OUT / 'event_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if not catalog.empty:
    top_plants = (
        catalog.groupby(['plant', 'plant_id'])
        .agg(
            n_events=('method', 'size'),
            n_methods=('method', 'nunique'),
            n_include=('action_for_training', lambda x: (x == 'include').sum()),
            n_downweight=('action_for_training', lambda x: (x == 'downweight').sum()),
            n_exclude=('action_for_training', lambda x: (x == 'exclude').sum()),
            first_event=('date', 'min'),
            last_event=('date', 'max'),
            max_severity=('severity', 'max'),
        )
        .sort_values('n_events', ascending=False)
        .reset_index()
    )
    print('Top 20 plants by number of events:')
    display(top_plants.head(20))
    top_plants.to_csv(OUT / 'plant_event_summary.csv', index=False)

    print(f'\n=== action_for_training distribution (pie) ===')
    action_counts = catalog['action_for_training'].value_counts()
    fig, ax = plt.subplots(figsize=(6, 6))
    colors = {'include': '#4CAF50', 'downweight': '#FFC107', 'exclude': '#F44336'}
    ax.pie(action_counts, labels=action_counts.index, autopct='%1.1f%%',
           colors=[colors.get(a, '#999') for a in action_counts.index])
    ax.set_title('Event catalog: action_for_training')
    plt.tight_layout()
    fig.savefig(OUT / 'action_for_training_pie.png', dpi=150, bbox_inches='tight')
    plt.show()

## Lettura per la tesi

Questo notebook rileva **eventi puntuali di produzione** (anomalie isolate in singoli mesi)
che il trend analysis non cattura.

**Classificazione per continual learning.** Ogni evento ha `action_for_training`:
- `include`: evento ambientale plausibile. Il modello deve imparare che con heatwave si produce meno.
- `downweight`: ambiguo. Potrebbe essere reale o artefatto. Peso ridotto nel training.
- `exclude`: quasi certamente equipment failure o data issue. Escludere dal training per non
  corrompere il modello (es. inverter trip che causa PR -50% non e' pattern meteorologico).

La classificazione usa due criteri (letteratura NREL):
1. **Severita'**: drop <15% = ambientale plausibile, 15-25% = zona grigia, >25% = equipment/data.
2. **Fleet context**: se molti impianti hanno anomalia nello stesso mese = causa condivisa
   (meteo), la soglia di esclusione e' piu' alta.

L'event catalog CSV (`event_catalog.csv`) puo' essere usato come:
- **Sample weighting mask** per continual learning: la colonna `action_for_training` indica
  direttamente come trattare ogni (plant, month) nel training.
- **Post-hoc error analysis**: se il modello sbaglia su un mese, controllare se e' nel catalogo.

## Report compatto

Esegui questa cella per generare un riepilogo testuale completo. Copia l'output e incollalo per analisi esterna.

In [ ]:
_lines = []
_lines.append("=" * 70)
_lines.append("PUNCTUAL EVENT DETECTION — COMPACT REPORT")
_lines.append("=" * 70)

# --- Thresholds & filters ---
_lines.append(f"\nThresholds: modified_zscore={ZSCORE_THRESHOLD}, "
              f"MoM_drop={MOM_DROP_PCT}%, MoM_spike={MOM_SPIKE_PCT}%, "
              f"fleet_zscore={FLEET_ZSCORE_THRESHOLD}")
_lines.append(f"Severity: moderate_drop={MODERATE_DROP_PCT}%, "
              f"severe_drop={SEVERE_DROP_PCT}% (NREL-based)")
_lines.append(f"Filters: edge_months=dropped, min_valid_hours={MIN_VALID_HOURS}, "
              f"PR_max={PR_PLAUSIBLE_MAX}")
_lines.append(f"Co-occurrence: {COOCCURRENCE_MIN_PLANTS} plants "
              f"(max({COOCCURRENCE_MIN_FLOOR}, {n_fleet_plants}*{COOCCURRENCE_PCT:.0%}))")
_lines.append(f"Data after filters: {plant_monthly['plant'].nunique()} plants, "
              f"{len(plant_monthly)} rows, "
              f"{plant_monthly['date'].min().strftime('%Y-%m')} to "
              f"{plant_monthly['date'].max().strftime('%Y-%m')}")

# --- 1. Modified Z-score ---
_lines.append("\n" + "=" * 70)
_lines.append("1. PLANT-LEVEL POINT ANOMALIES (modified Z-score)")
_lines.append("=" * 70)
if anomaly_zscore.empty:
    _lines.append("No anomalies found.")
else:
    _lines.append(f"Total anomalies: {len(anomaly_zscore)}")
    _lines.append(f"Plants affected: {anomaly_zscore['plant'].nunique()}")
    _lines.append(f"\nBy event type:")
    for et, cnt in anomaly_zscore.groupby('event_type').size().items():
        _lines.append(f"  {et}: {cnt}")
    _lines.append(f"\nTop 15 most severe (sorted by |Z|):")
    top_z = anomaly_zscore.reindex(anomaly_zscore['modified_zscore'].abs().sort_values(ascending=False).index).head(15)
    for _, r in top_z.iterrows():
        _lines.append(
            f"  plant={r['plant_id']}  date={r['date'].strftime('%Y-%m') if hasattr(r['date'], 'strftime') else r['date']}  "
            f"PR={r['pr_pvgis']:.3f}  median={r['plant_median_pr']:.3f}  Z={r['modified_zscore']:.2f}  type={r['event_type']}"
        )

# --- 2. MoM shock ---
_lines.append("\n" + "=" * 70)
_lines.append("2. MONTH-OVER-MONTH SHOCKS")
_lines.append("=" * 70)
if anomaly_mom.empty:
    _lines.append("No MoM shocks found.")
else:
    _lines.append(f"Total shocks: {len(anomaly_mom)}")
    _lines.append(f"Transient (recovery): {int(anomaly_mom['recovery'].sum())}")
    _lines.append(f"Sustained (no recovery): {int((~anomaly_mom['recovery']).sum())}")
    _lines.append(f"\nBy event type:")
    for et, cnt in anomaly_mom.groupby('event_type').size().items():
        _lines.append(f"  {et}: {cnt}")
    _lines.append(f"\nTop 15 largest drops:")
    top_mom = anomaly_mom.sort_values('mom_change_pct').head(15)
    for _, r in top_mom.iterrows():
        _lines.append(
            f"  plant={r['plant_id']}  date={r['date'].strftime('%Y-%m') if hasattr(r['date'], 'strftime') else r['date']}  "
            f"PR={r['pr_pvgis']:.3f}  prev={r['pr_pvgis_prev']:.3f}  "
            f"MoM={r['mom_change_pct']:.1f}%  recovery={r['recovery']}  type={r['event_type']}"
        )

# --- 3. Fleet-relative ---
_lines.append("\n" + "=" * 70)
_lines.append("3. FLEET-RELATIVE OUTLIERS")
_lines.append("=" * 70)
if anomaly_fleet.empty:
    _lines.append("No fleet-relative outliers found.")
else:
    _lines.append(f"Total outliers: {len(anomaly_fleet)}")
    _lines.append(f"Plants flagged: {anomaly_fleet['plant'].nunique()}")
    _lines.append(f"\nBy event type:")
    for et, cnt in anomaly_fleet.groupby('event_type').size().items():
        _lines.append(f"  {et}: {cnt}")
    _lines.append(f"\nTop 15 most deviant (below fleet):")
    top_fl = anomaly_fleet.sort_values('fleet_zscore').head(15)
    for _, r in top_fl.iterrows():
        _lines.append(
            f"  plant={r['plant_id']}  date={r['date'].strftime('%Y-%m') if hasattr(r['date'], 'strftime') else r['date']}  "
            f"PR={r['pr_pvgis']:.3f}  fleet_median={r['fleet_median_pr']:.3f}  fleet_Z={r['fleet_zscore']:.2f}"
        )

# --- 4. Co-occurrence ---
_lines.append("\n" + "=" * 70)
_lines.append("4. FLEET CO-OCCURRENCE")
_lines.append("=" * 70)
if cooccurrence.empty:
    _lines.append("No co-occurrence data.")
else:
    _lines.append(f"Months with >= {COOCCURRENCE_MIN_PLANTS} plants flagged: {len(fleet_events)}")
    _lines.append(f"\nAll months ranked by n_plants_flagged:")
    for _, r in cooccurrence.iterrows():
        dt_str = r['date'].strftime('%Y-%m') if hasattr(r['date'], 'strftime') else str(r['date'])
        marker = " *** FLEET EVENT" if r['n_plants_flagged'] >= COOCCURRENCE_MIN_PLANTS else ""
        _lines.append(f"  {dt_str}: {r['n_plants_flagged']} plants{marker}")

# --- 5. Catalog + training action ---
_lines.append("\n" + "=" * 70)
_lines.append("5. EVENT CATALOG + TRAINING ACTION")
_lines.append("=" * 70)
if catalog.empty:
    _lines.append("Catalog empty.")
else:
    _lines.append(f"Total events: {len(catalog)}")
    _lines.append(f"Unique plants: {catalog['plant'].nunique()}")
    _lines.append(f"Date range: {catalog['date'].min()} to {catalog['date'].max()}")

    _lines.append(f"\n--- action_for_training ---")
    for action, cnt in catalog['action_for_training'].value_counts().items():
        pct = cnt / len(catalog) * 100
        _lines.append(f"  {action}: {cnt} ({pct:.1f}%)")

    _lines.append(f"\n--- action_for_training by method ---")
    cross_action = catalog.groupby(['method', 'action_for_training']).size().unstack(fill_value=0)
    _lines.append(cross_action.to_string())

    _lines.append(f"\n--- method x event_type ---")
    cross = catalog.groupby(['method', 'event_type']).size().unstack(fill_value=0)
    _lines.append(cross.to_string())

    _lines.append(f"\nTop 20 plants by event count:")
    top_p = (
        catalog.groupby(['plant', 'plant_id'])
        .agg(n_events=('method', 'size'), n_methods=('method', 'nunique'),
             n_exclude=('action_for_training', lambda x: (x == 'exclude').sum()),
             n_downweight=('action_for_training', lambda x: (x == 'downweight').sum()),
             max_severity=('severity', 'max'))
        .sort_values('n_events', ascending=False)
        .head(20)
        .reset_index()
    )
    for _, r in top_p.iterrows():
        _lines.append(
            f"  plant={r['plant_id']}  events={r['n_events']}  "
            f"methods={r['n_methods']}  exclude={int(r['n_exclude'])}  "
            f"downweight={int(r['n_downweight'])}  max_severity={r['max_severity']:.2f}"
        )

    multi = catalog.groupby('plant')['method'].nunique()
    _lines.append(f"\nPlants flagged by 1 method only: {(multi == 1).sum()}")
    _lines.append(f"Plants flagged by 2 methods: {(multi == 2).sum()}")
    _lines.append(f"Plants flagged by 3 methods: {(multi == 3).sum()}")

# --- Interpretation ---
_lines.append("\n" + "=" * 70)
_lines.append("INTERPRETATION NOTES")
_lines.append("=" * 70)
_lines.append(
    "- Focus: production events only (PR_PVGIS). QS/data diagnostics in\n"
    "  domain_shift_trend_analysis.ipynb.\n"
    "- Edge months (first/last per plant) excluded: partial data inflates PR.\n"
    f"- Months with <{MIN_VALID_HOURS} valid hours excluded.\n"
    f"- Months with PR>{PR_PLAUSIBLE_MAX} excluded (physically impossible).\n"
    "- 'transient' = drop/spike that recovers next month (>=50% reversal).\n"
    "- 'sustained' in MoM context = shock without recovery.\n"
    f"\n- action_for_training classification (NREL-based thresholds):\n"
    f"    include:    deviation <{MODERATE_DROP_PCT}%, OR fleet-wide + <{SEVERE_DROP_PCT}%\n"
    f"    downweight: plant-specific 15-25%, OR fleet-wide >{SEVERE_DROP_PCT}%\n"
    f"    exclude:    plant-specific >{SEVERE_DROP_PCT}% (equipment/data issue)\n"
    "- Fleet co-occurrence: threshold is proportional to fleet size.\n"
    "  Fleet-wide months get softer classification (shared environmental cause).\n"
    "- Event catalog saved to: outputs/punctual_events/event_catalog.csv\n"
    "- Plant summary saved to: outputs/punctual_events/plant_event_summary.csv"
)

_report = "\n".join(_lines)
print(_report)